# Sistema de Evaluación de Riesgo de Bajo Rendimiento Académico
## Institución Universitaria Pascual Bravo · Medellín, Colombia
### Flujo: Delphi → Sistema Difuso Mamdani → Simulación Montecarlo → Regresión


In [ ]:
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Agregar el directorio raíz al path
sys.path.insert(0, os.path.abspath('..'))

# Semilla global para reproducibilidad
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Verificar disponibilidad de paquetes
required_packages = ['pandas', 'numpy', 'matplotlib', 'skfuzzy', 'sklearn', 'scipy', 'hypothesis']
for pkg in required_packages:
    try:
        __import__(pkg)
        print(f"\u2713 {pkg} disponible")
    except ImportError:
        raise ImportError(f"Paquete '{pkg}' no encontrado. Instalar con: pip install -r requirements.txt")

print("\n\u2713 Todos los paquetes requeridos están disponibles")
print(f"\u2713 RANDOM_SEED = {RANDOM_SEED}")


---
## Parte A: Método Delphi
Simulación del proceso Delphi con panel de expertos de la Institución Universitaria Pascual Bravo.


In [ ]:
from delphi import ExpertPanel, DelphiSimulator

# Instanciar panel y simulador
panel = ExpertPanel(seed=RANDOM_SEED)
simulator = DelphiSimulator(panel, data_dir='../data/', docs_dir='../docs/')

# Ejecutar las 3 rondas
print("Ejecutando Ronda 1 (Exploración)...")
r1 = simulator.run_round1()
print(f"  Factores evaluados: {[f['factor'] for f in r1['factores']]}")

print("\nEjecutando Ronda 2 (Priorización)...")
r2 = simulator.run_round2(r1)

print("\nEjecutando Ronda 3 (Validación y Consenso)...")
r3 = simulator.run_round3(r2)

# Mostrar resultados
print("\n=== VARIABLES APROBADAS ===")
for v in r3['variables_aprobadas']:
    st = v['estadisticos_finales']
    print(f"  \u2713 {v['factor']}: media={st['media']:.3f}, CV={st['cv']:.3f}, aprobación={v['criterios']['approval_pct']:.0f}%")

print(f"\n  Variables rechazadas: {len(r3['variables_rechazadas'])}")
print("\n\u2713 Proceso Delphi completado. Archivos generados en data/ y docs/")


In [ ]:
import json

# Verificar trazabilidad: todas las Variables_Entrada deben estar en delphi_consenso.json
with open('../data/delphi_consenso.json', encoding='utf-8') as f:
    consenso = json.load(f)

variables_aprobadas = {v['factor'] for v in consenso['variables_aprobadas']}
variables_esperadas = {'promedio_academico', 'inasistencia', 'horas_estudio', 'motivacion_estres'}

print("=== VERIFICACIÓN DE TRAZABILIDAD ===")
for var in sorted(variables_esperadas):
    status = "\u2713" if var in variables_aprobadas else "\u2717"
    print(f"  {status} {var}")

assert variables_esperadas.issubset(variables_aprobadas), \
    f"Variables faltantes en consenso: {variables_esperadas - variables_aprobadas}"

print("\n\u2713 Trazabilidad verificada: todas las variables del sistema difuso están en el consenso Delphi")


---
## Parte B: Sistema de Inferencia Difuso Mamdani
Construcción del sistema difuso con variables y reglas derivadas del consenso Delphi.


In [ ]:
from fuzzy_system import FuzzySystemBuilder

# Construir sistema difuso
print("Construyendo sistema difuso Mamdani...")
fs = FuzzySystemBuilder(
    consenso_path='../data/delphi_consenso.json',
    data_dir='../data/',
    docs_dir='../docs/'
)
fs.build()
print("\u2713 Sistema difuso construido")
print(f"  Variables de entrada: {list(fs.UNIVERSES.keys())[:-1]}")
print(f"  Variable de salida: riesgo [0, 100]")
print(f"  Defuzzificación: centroide")

# Mostrar gráficas de funciones de pertenencia
from IPython.display import Image, display
import os

plots_dir = '../docs/fuzzy_membership_plots'
for fname in sorted(os.listdir(plots_dir)):
    if fname.endswith('.png'):
        print(f"\n  Funciones de pertenencia: {fname.replace('.png', '')}")
        display(Image(filename=os.path.join(plots_dir, fname)))

# Prueba del sistema
print("\n=== PRUEBA DEL SISTEMA DIFUSO ===")
casos_prueba = [
    {"promedio_academico": 2.0, "inasistencia": 65.0, "horas_estudio": 4.0, "motivacion_estres": 2.0},
    {"promedio_academico": 4.5, "inasistencia": 5.0, "horas_estudio": 25.0, "motivacion_estres": 8.0},
    {"promedio_academico": 3.5, "inasistencia": 35.0, "horas_estudio": 12.0, "motivacion_estres": 5.0},
]
etiquetas = ["Alto riesgo esperado", "Bajo riesgo esperado", "Riesgo medio esperado"]
for caso, etiqueta in zip(casos_prueba, etiquetas):
    riesgo = fs.evaluar_riesgo(caso)
    print(f"  {etiqueta}: riesgo = {riesgo:.2f}")


---
## Parte C: Simulación Montecarlo
Generación de 1000 escenarios aleatorios para estimar la distribución del riesgo académico.


In [ ]:
from montecarlo import MontecarloSimulator

# Ejecutar simulación
print("Ejecutando simulación Montecarlo (1000 simulaciones)...")
mc = MontecarloSimulator(fs, data_dir='../data/', docs_dir='../docs/', seed=RANDOM_SEED)
df = mc.run(n_simulaciones=1000)

# Mostrar estadísticas
stats = mc._statistics
print(f"\n=== ESTADÍSTICAS DEL RIESGO SIMULADO ===")
print(f"  Media:          {stats['mean']:.2f}")
print(f"  Desv. estándar: {stats['std']:.2f}")
print(f"  Mínimo:         {stats['min']:.2f}")
print(f"  P25:            {stats['p25']:.2f}")
print(f"  Mediana (P50):  {stats['p50']:.2f}")
print(f"  P75:            {stats['p75']:.2f}")
print(f"  P95:            {stats['p95']:.2f}")
print(f"  Máximo:         {stats['max']:.2f}")
print(f"  P(riesgo \u2265 70): {stats['p_riesgo_alto']:.1%}")

n_criticos = len(mc._critical_scenarios)
print(f"\n  Escenarios críticos (riesgo \u2265 70): {n_criticos} ({n_criticos/len(df):.1%})")

# Mostrar histograma
display(Image(filename='../docs/montecarlo_histograma.png'))
print(f"\n\u2713 Base simulada guardada: {len(df)} filas en data/base_simulada.csv")


---
## Parte D: Regresión y Análisis Comparativo
Entrenamiento de modelos predictivos sobre la base simulada y comparación con el sistema difuso.


In [ ]:
from regression import RegressionAnalyzer

# Entrenar y evaluar modelos
print("Entrenando modelos de regresión...")
ra = RegressionAnalyzer(
    data_path='../data/base_simulada.csv',
    consenso_path='../data/delphi_consenso.json',
    docs_dir='../docs/'
)

metrics = ra.train_and_evaluate()

print("\n=== MÉTRICAS DE EVALUACIÓN ===")
print(f"{'Modelo':<20} {'MAE':>8} {'RMSE':>8} {'R\u00b2':>8}")
print("-" * 46)
model_names = {"knn": "KNN (k=5)", "random_forest": "Random Forest", "decision_tree": "Decision Tree"}
for name, m in metrics.items():
    print(f"{model_names[name]:<20} {m['mae']:>8.4f} {m['rmse']:>8.4f} {m['r2']:>8.4f}")

# Importancia de variables
importance = ra.get_feature_importance()
print("\n=== IMPORTANCIA DE VARIABLES (Random Forest) ===")
rf_imp = sorted(importance['random_forest'].items(), key=lambda x: x[1], reverse=True)
for var, val in rf_imp:
    bar = "\u2588" * int(val * 40)
    print(f"  {var:<25} {val:.4f} {bar}")

# Correlación de Pearson
corr = ra.calculate_pearson_correlation()
print(f"\n=== CORRELACIÓN DE PEARSON ===")
print(f"  Mejor modelo vs. valores difusos: r = {corr:.4f}")

# Generar todos los reportes
ra.generate_scatter_plot()
ra.generate_comparative_report(metrics)
ra.generate_importance_report()
ra.generate_comparative_analysis(metrics, corr)

# Mostrar gráfico de dispersión
display(Image(filename='../docs/comparativo_difuso_vs_prediccion.png'))
print("\n\u2713 Todos los reportes de regresión generados en docs/")


In [ ]:
import os

print("=" * 60)
print("RESUMEN FINAL \u2014 EJECUCIÓN COMPLETADA")
print("=" * 60)

archivos_data = [
    'data/delphi_ronda1.json',
    'data/delphi_ronda2.json',
    'data/delphi_consenso.json',
    'data/fuzzy_variables.json',
    'data/fuzzy_rules.json',
    'data/base_simulada.csv',
]

archivos_docs = [
    'docs/delphi_informe.md',
    'docs/montecarlo_distribuciones.md',
    'docs/montecarlo_histograma.png',
    'docs/regression_comparativa.md',
    'docs/regression_importancia_variables.md',
    'docs/analisis_comparativo.md',
    'docs/comparativo_difuso_vs_prediccion.png',
    'docs/trazabilidad.md',
]

print("\nArchivos de datos generados:")
for f in archivos_data:
    path = f'../{f}'
    status = "\u2713" if os.path.exists(path) else "\u2717 FALTANTE"
    print(f"  {status} {f}")

print("\nDocumentos generados:")
for f in archivos_docs:
    path = f'../{f}'
    status = "\u2713" if os.path.exists(path) else "\u2717 FALTANTE"
    print(f"  {status} {f}")

print("\n\u2713 Flujo completo ejecutado: Delphi \u2192 Difuso \u2192 Montecarlo \u2192 Regresión")
print(f"\u2713 RANDOM_SEED = {RANDOM_SEED} \u2014 Resultados reproducibles")
